# 8.6.循环神经网络的简洁实现

虽然 [8.5节](./08.05_rnn_scratch.ipynb)对了解循环神经网络的实现方式具有指导意义，但并不方便。本节将展示如何使用深度学习框架的高级API提供的函数更有效地实现相同的语言模型。我们仍然从读取时光机器数据集开始。


## 环境配置

In [2]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor with interal format")
import pypto
import torch
from torch import nn
from torch.nn import functional as F
import torch_npu
import logging
logging.getLogger('matplotlib').setLevel(logging.WARNING)
logging.getLogger('torch_npu.env').setLevel(logging.ERROR)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()


In [3]:
from src.utils import load_data_time_machine, train_ch8, predict_ch8
from src.pypto_ops import PyPTOLinear, PyPTORNN, loss_fn

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)
</pre>
  </div>
</details>


## 8.6.1.定义模型

高级API提供了循环神经网络的实现。我们构造一个具有256个隐藏单元的单隐藏层的循环神经网络层`rnn_layer`。事实上，我们还没有讨论多层循环神经网络的意义（这将在 9.3节深度循环神经网络中介绍）。现在仅需要将多层理解为一层循环神经网络的输出被用作下一层循环神经网络的输入就足够了。


In [4]:
num_hiddens = 256
rnn_layer = PyPTORNN(len(vocab), num_hiddens)

我们**使用张量来初始化隐状态**，它的形状是（隐藏层数，批量大小，隐藏单元数）。


In [5]:
state = torch.zeros((1, batch_size, num_hiddens))
state.shape

torch.Size([1, 32, 256])

**通过一个隐状态和一个输入，我们就可以用更新后的隐状态计算输出。** 需要强调的是，`rnn_layer`的“输出”（`Y`）不涉及输出层的计算：它是指每个时间步的隐状态，这些隐状态可以用作后续输出层的输入。


In [6]:
num_inputs = len(vocab)
rnn_layer = rnn_layer.to(device)
X = torch.rand(size=(num_steps, batch_size, num_inputs), device=device)
state = torch.zeros(size=(1, batch_size, num_hiddens), device=device)
Y, state_new = rnn_layer(X, state)
Y.shape, state_new.shape

(torch.Size([35, 32, 256]), torch.Size([1, 32, 256]))

此外，`rnn_layer`返回的更新后的隐状态（`state_new`）是指小批量数据的最后时间步的隐状态。这个隐状态可以用来初始化顺序分区中一个迭代周期内下一个小批量数据的隐状态。对于多个隐藏层，每一层的隐状态将存储在（`state_new`）变量中。至于稍后要介绍的某些模型（例如，长－短期记忆），此变量还包含其他信息。


与 [8.5节](./08.05_rnn_scratch.ipynb)类似，**我们为一个完整的循环神经网络模型定义了一个`RNNModel`类**。注意，`rnn_layer`只包含隐藏的循环层，我们还需要创建一个单独的输出层。


In [7]:
class PyPTORNNModel(nn.Module):
    """基于 PyPTORNN + PyPTOLinear 的 RNN 模型。

    隐藏层使用 PyPTO 实现的 PyPTORNN，输出层使用 PyPTOLinear。
    """

    def __init__(self, rnn_layer, vocab_size, device=None):
        super().__init__()
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size

        if not self.rnn.bidirectional:
            self.num_directions = 1
            out_features = self.num_hiddens
        else:
            self.num_directions = 2
            out_features = self.num_hiddens * 2

        self.linear = PyPTOLinear(out_features, self.vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(device=inputs.device, dtype=torch.float32)
        Y, state = self.rnn(X, state)

        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, device=None, batch_size=1):
        if device is None:
            device = next(self.parameters()).device
        return torch.zeros(
            self.num_directions * self.rnn.num_layers,
            batch_size, self.num_hiddens, device=device)

In [8]:
net = PyPTORNNModel(rnn_layer, len(vocab)).to(device)

# 检查输出形状
X, y = next(iter(train_iter))
X, y = X.to(device), y.to(device)
state = net.begin_state(device=device, batch_size=X.shape[0])
Y, new_state = net(X, state)
Y.shape, new_state.shape

(torch.Size([1120, 28]), torch.Size([1, 32, 256]))

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_hiddens = 256
rnn_layer = nn.RNN(len(vocab), num_hiddens)
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
state = torch.zeros((1, batch_size, num_hiddens))
state.shape
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
X = torch.rand(size=(num_steps, batch_size, len(vocab)))
Y, state_new = rnn_layer(X, state)
Y.shape, state_new.shape
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class RNNModel(nn.Module):
    """循环神经网络模型"""
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size
        # 如果RNN是双向的（之后将介绍），num_directions应该是2，否则应该是1
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)
        Y, state = self.rnn(X, state)
        # 全连接层首先将Y的形状改为(时间步数*批量大小,隐藏单元数)
        # 它的输出形状是(时间步数*批量大小,词表大小)。
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, device, batch_size=1):
        if not isinstance(self.rnn, nn.LSTM):
            # nn.GRU以张量作为隐状态
            return torch.zeros((self.num_directions * self.rnn.num_layers,
                                 batch_size, self.num_hiddens),
                                device=device)
        else:
            # nn.LSTM以元组作为隐状态
            return (torch.zeros((
                self.num_directions * self.rnn.num_layers,
                batch_size, self.num_hiddens), device=device),
                    torch.zeros((
                        self.num_directions * self.rnn.num_layers,
                        batch_size, self.num_hiddens), device=device))
</pre>
  </div>
</details>


## 8.6.2.训练与预测

在训练模型之前，让我们**基于一个具有随机权重的模型进行预测**。


In [9]:
# 训练前的预测（应该是乱码）
print(predict_ch8('time traveller ', 10, net, vocab, device))

time traveller prftqbdczb


很明显，这种模型根本不能输出好的结果。接下来，我们使用 [8.5节](./08.05_rnn_scratch.ipynb)中定义的超参数调用`train_ch8`，并且**使用高级API训练模型**。


In [10]:
# 预热：触发 PyPTO kernel（RNN 隐层 + 输出层）的首次编译
print('正在编译 pypto kernel（首次运行需约半分钟，请耐心等待）...')
import time; t0 = time.time()
# 取一个批次触发 JIT 编译
X, y = next(iter(train_iter))
X, y = X.to(device), y.to(device)
state = net.begin_state(device=device, batch_size=X.shape[0])
Y, new_state = net(X, state)
y = y.T.reshape(-1).to(device)
# 使用 PyPTO loss_fn：softmax + CE 全程在 NPU 上执行
l = loss_fn(Y, y.long(), num_classes=len(vocab))
l.backward()
# 重置梯度，为正式训练做准备
net.zero_grad()
print(f'编译完成，耗时 {time.time()-t0:.0f} 秒。接下来可以正常训练了。')

正在编译 pypto kernel（首次运行需约半分钟，请耐心等待）...


编译完成，耗时 19 秒。接下来可以正常训练了。


In [11]:
num_epochs, lr = 500, 1
train_ch8(net, train_iter, vocab, lr, num_epochs, device, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

time travellere the the the the the the the the the the the the 
  [epoch 10/500] perplexity=15.1


time travellere the the the the the the the the the the the the 
  [epoch 20/500] perplexity=10.9


time travellere the the the the the the the the the the the the 
  [epoch 30/500] perplexity=10.0


time travellere the the the the the the the the the the the the 
  [epoch 40/500] perplexity=9.3


time traveller the the the the the the the the the the the the t
  [epoch 50/500] perplexity=8.7


...

time travellerit would be remarkably upare winn wo als ghath der
  [epoch 460/500] perplexity=1.3


time traveller thicee a somed al mayouby bat some hion whis whil
  [epoch 470/500] perplexity=1.3


time traveller but now you begint bo as acceduld is onsedowe sig
  [epoch 480/500] perplexity=1.3


time traveller cime sime an wh che inctions and mery wila wanis 
  [epoch 490/500] perplexity=1.2


time travellerit would be remarkably convenient for the historia
  [epoch 500/500] perplexity=1.3
困惑度 1.3, 25253.6 词元/秒 npu:0
time travellerit would be remarkably convenient for the historia
travelleryou can show blackensions of space but you cannotm


我们还可以使用随机抽样方法来训练模型，并比较两种方法的困惑度差异。


In [12]:
net_random = PyPTORNNModel(
    PyPTORNN(len(vocab), num_hiddens), len(vocab)).to(device)
train_ch8(net_random, train_iter, vocab, lr, num_epochs, device,
          use_random_iter=True, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

time traveller t t t t t t t t t t t t t t t t t t t t t t t t t
  [epoch 10/500] perplexity=14.8


time travellere the the the the the the the the the the the the 
  [epoch 20/500] perplexity=11.0


time traveller the the the the the the the the the the the the t
  [epoch 30/500] perplexity=10.0


time traveller the the the the the the the the the the the the t
  [epoch 40/500] perplexity=9.4


time traveller the the the the the the the the the the the the t
  [epoch 50/500] perplexity=8.9


...

time traveller held in his hand was a glitteringmetallic fremers
  [epoch 470/500] perplexity=1.7


time travellerit soall hemether have to thepsthree which wr a mo
  [epoch 480/500] perplexity=1.7


time traveller held in his hand a socethot mane there are balloo
  [epoch 490/500] perplexity=1.5


time traveller provence vere you spack and his usuane this free 
  [epoch 500/500] perplexity=1.7
困惑度 1.7, 23300.9 词元/秒 npu:0
time traveller provence vere you spack and his usuane this free 


traveller heldiceds tnat soallact reever its i neethonova i


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
device = d2l.try_gpu()
net = RNNModel(rnn_layer, vocab_size=len(vocab))
net = net.to(device)
d2l.predict_ch8('time traveller', 10, net, vocab, device)
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_epochs, lr = 500, 1
d2l.train_ch8(net, train_iter, vocab, lr, num_epochs, device)
</pre>
  </div>
</details>


## 8.6.3.小结

* 深度学习框架的高级API提供了循环神经网络层的实现。

* 高级API的循环神经网络层返回一个输出和一个更新后的隐状态，我们还需要计算整个模型的输出层。

* 相比从零开始实现的循环神经网络，使用高级API实现可以加速训练。

## 8.6.4.练习

1. 尝试使用高级API，能使循环神经网络模型过拟合吗？

1. 如果在循环神经网络模型中增加隐藏层的数量会发生什么？能使模型正常工作吗？

1. 尝试使用循环神经网络实现 [8.1节](./08.01_sequence.ipynb)的自回归模型。

参考答案详见 [answers/08.06_reference_answer](./answers/08.06_reference_answer.ipynb)。


### 参考答案（PyPTO）

In [ ]:
!cat answers/txt/08.06_reference_answer_pypto.txt

### 参考答案（PyTorch）

In [ ]:
!cat answers/txt/08.06_reference_answer_pytorch.txt